# Huấn luyện HPMA trên Google Colab (L4)

Notebook này dùng pipeline trong repository. ISLES-2022 được chia **theo bệnh nhân/case**, với mục tiêu 80% train, 10% validation, 10% test; mọi lát của một case ở cùng một split. Tỷ lệ thực tế theo số case có thể lệch nhẹ do số case là số nguyên.

`train.py` đánh giá validation mỗi epoch và lưu `checkpoints/hpma_isles-2022_best.pt` khi `mDice` validation tăng. Test chỉ chạy sau khi đã hoàn tất lựa chọn best checkpoint; không dùng test để chọn epoch hay tune cấu hình.

Notebook đặt `system.device` thành `cuda:0`. DataLoader đọc dữ liệu từ CPU theo batch; mọi tensor ảnh/mask/label của batch được chuyển sang GPU 0 trước khi model hoặc memory builder xử lý. Không nạp toàn bộ dataset vào VRAM.


## 1. Kiểm tra GPU

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError("Hãy chọn Runtime có GPU trong Runtime > Change runtime type.")

torch.cuda.set_device(0)
DEVICE = torch.device('cuda:0')
print(f"GPU: {torch.cuda.get_device_name(0)} ({DEVICE})")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch.backends.cuda.matmul.allow_tf32 = True

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Lấy code và cài dependencies

In [ ]:
import os
from pathlib import Path

REPO_DIR = Path('/content/StrokeSAM')
if REPO_DIR.exists():
    %cd /content/StrokeSAM
    !git pull --ff-only
else:
    %cd /content
    !git clone https://github.com/hoangtung386/StrokeSAM.git
    %cd /content/StrokeSAM

In [ ]:
!pip install -q -e '.[dev]'
!pip install -q matplotlib

## 4. Dataset và SAM 3.1 checkpoint

Đặt ISLES-2022 đã giải nén trong Drive theo cấu trúc BIDS mà loader yêu cầu (`sub-*/ses-*/dwi`, `anat`, và `derivatives`). Có thể sửa `DRIVE_DATASET_DIR` bên dưới. Checkpoint SAM 3.1 cần được đặt trong Drive. Môi trường cũng phải có package `sam3` chính thức tương thích với checkpoint; notebook dừng sớm nếu thiếu package hoặc checkpoint thay vì chạy tiếp rồi lỗi ở giữa pipeline.


In [ ]:
import shutil

DRIVE_DATASET_DIR = Path('/content/drive/MyDrive/Stroke/ISLES-2022')
DATASET_DIR = REPO_DIR / 'dataset' / 'ISLES-2022'
DRIVE_SAM_CHECKPOINT = Path('/content/drive/MyDrive/checkpoints/sam3.1.pt')
SAM_CHECKPOINT = REPO_DIR / 'checkpoints' / 'sam3.1.pt'

# False: truy cập trực tiếp file trên Drive, không chép lại cả dataset mỗi lần chạy.
# True: chép dataset vào ổ Colab một lần để tăng tốc đọc dữ liệu khi train.
COPY_DATASET_TO_LOCAL = False


In [ ]:
# Chỉ import để xác nhận package; dòng này không khởi tạo model hay load weights.
try:
    import sam3
except ImportError as exc:
    raise ImportError(
        "Thiếu package SAM 3 chính thức. Cài package SAM 3 trước khi chạy notebook."
    ) from exc

if not DRIVE_DATASET_DIR.is_dir():
    raise FileNotFoundError(f"Không tìm thấy dataset folder: {DRIVE_DATASET_DIR}")
if not DRIVE_SAM_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Không tìm thấy SAM checkpoint: {DRIVE_SAM_CHECKPOINT}")

DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
if DATASET_DIR.is_symlink():
    DATASET_DIR.unlink()

if DATASET_DIR.exists() and not COPY_DATASET_TO_LOCAL:
    if not next(DATASET_DIR.glob('sub-*/ses-*/dwi/*_dwi.nii.gz'), None):
        raise FileNotFoundError(f"Thư mục dataset local không có case ISLES hợp lệ: {DATASET_DIR}")
    print(f"Dùng lại dataset đã có trên ổ Colab: {DATASET_DIR}")
elif COPY_DATASET_TO_LOCAL:
    if DATASET_DIR.exists():
        shutil.rmtree(DATASET_DIR)
    shutil.copytree(DRIVE_DATASET_DIR, DATASET_DIR)
else:
    DATASET_DIR.symlink_to(DRIVE_DATASET_DIR, target_is_directory=True)

SAM_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if SAM_CHECKPOINT.is_symlink() or SAM_CHECKPOINT.is_file():
    SAM_CHECKPOINT.unlink()
SAM_CHECKPOINT.symlink_to(DRIVE_SAM_CHECKPOINT)

print(f"Dataset path: {DATASET_DIR}")
print(f"SAM checkpoint path: {SAM_CHECKPOINT}")
print("Đã kiểm tra package sam3; model chỉ được tạo khi build memory/train/eval.")


In [ ]:
print(f"Dataset: {DATASET_DIR}")
print(f"SAM checkpoint: {SAM_CHECKPOINT}")

## 5. Cấu hình tỷ lệ split và huấn luyện

In [ ]:
import yaml

BASE_CONFIG = REPO_DIR / 'configs' / 'base_config.yaml'
DATA_CONFIG = REPO_DIR / 'configs' / 'isles2022.yaml'
with DATA_CONFIG.open(encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)
with BASE_CONFIG.open(encoding='utf-8') as f:
    base_cfg = yaml.safe_load(f)

data_cfg['dataset']['data_root'] = str(DATASET_DIR)
data_cfg['dataset']['validation_fraction'] = 0.10
data_cfg['dataset']['test_fraction'] = 0.10
base_cfg['system']['device'] = 'cuda:0'
base_cfg['system']['seed'] = 42
base_cfg['system']['num_workers'] = 2
base_cfg['model']['sam3_checkpoint'] = str(SAM_CHECKPOINT)
# Cấu hình thận trọng cho L4 24 GB; giảm batch xuống 1 nếu vẫn thiếu VRAM.
base_cfg['training']['batch_size'] = 1
base_cfg['training']['gradient_accumulation_steps'] = 8
base_cfg['training']['mixed_precision'] = True

In [ ]:
with DATA_CONFIG.open('w', encoding='utf-8') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
with BASE_CONFIG.open('w', encoding='utf-8') as f:
    yaml.safe_dump(base_cfg, f, sort_keys=False)

print('Đã cấu hình case split 80/10/10 và L4.')

## 6. Xác minh split theo case và trực quan hóa preprocessing

In [ ]:
import matplotlib.pyplot as plt
from data.factory import build_dataset

common = dict(
    dataset_config=data_cfg['dataset'],
    target_size=tuple(base_cfg['data']['input_resolution']),
    seed=base_cfg['system']['seed'],
    mean=tuple(base_cfg['data']['mean']),
    std=tuple(base_cfg['data']['std']),
)

datasets = {name: build_dataset(split=name, **common) for name in ('train', 'val', 'test')}
split_ids = {name: set(ds.case_splits[name]) for name, ds in datasets.items()}
all_ids = set.union(*split_ids.values())

assert not (split_ids['train'] & split_ids['val'])
assert not (split_ids['train'] & split_ids['test'])
assert not (split_ids['val'] & split_ids['test'])
assert set.union(*split_ids.values()) == all_ids

n_cases = len(all_ids)
for name in ('train', 'val', 'test'):
    ids = split_ids[name]
    print(f"{name:5s}: {len(ids):4d} cases ({100 * len(ids) / n_cases:.1f}%), "
          f"{len(datasets[name]):5d} 2D slices")

print(f"Tổng: {n_cases} cases. Mục tiêu 80/10/10; số case nguyên nên tỷ lệ có thể làm tròn.")

In [ ]:
sample = datasets['train'][0]
image, mask = sample['image'], sample['mask'][0]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, channel, title in zip(axes[:3], image, ('DWI', 'ADC', 'FLAIR')):
    ax.imshow(channel, cmap='gray')
    ax.set_title(title)
    ax.axis('off')

axes[3].imshow(image[0], cmap='gray')
axes[3].imshow(mask, cmap='Reds', alpha=0.35, vmin=0, vmax=1)
axes[3].set_title(f"DWI + mask | {sample['sample_id']}")
axes[3].axis('off')

plt.tight_layout()
plt.show()

## 7. Build prototype memory chỉ từ train

`build_memory.py` dựng memory bank từ train split duy nhất; validation và test không tham gia. Split được xác định lại bằng seed và fractions giống nhau trong các entrypoint.


In [ ]:
%cd /content/StrokeSAM
!python build_memory.py --config configs/isles2022.yaml --base_config configs/base_config.yaml

## 8. Train và chọn best model bằng validation mDice

In [ ]:
!python train.py --base_config configs/base_config.yaml --data_config configs/isles2022.yaml

`train.py` chạy validation sau mỗi epoch, ghi `Val Dice`/`Val IoU` vào `outputs/training.log`, và cập nhật file `*_best.pt` theo validation mDice. File `*_last.pt` là epoch cuối, không nhất thiết là model tốt nhất. Bước tiếp theo mới dùng test để báo cáo cuối cùng.


## 9. Đánh giá test sau khi đã cố định best checkpoint

In [ ]:
BEST_CHECKPOINT = REPO_DIR / 'checkpoints' / 'hpma_isles-2022_best.pt'
if not BEST_CHECKPOINT.is_file():
    raise FileNotFoundError(f"Không có best checkpoint: {BEST_CHECKPOINT}")

In [ ]:
!python eval.py --base_config configs/base_config.yaml --data_config configs/isles2022.yaml --checkpoint checkpoints/hpma_isles-2022_best.pt --split test

## 10. Sao lưu artifacts và thông tin run

In [ ]:
from datetime import datetime

BACKUP_DIR = Path('/content/drive/MyDrive/StrokeSAM_Run_Backup') / datetime.now().strftime('%Y%m%d_%H%M%S')
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
for source in (REPO_DIR / 'checkpoints', REPO_DIR / 'memory' / 'bank_cache', REPO_DIR / 'outputs'):
    if source.exists():
        shutil.copytree(source, BACKUP_DIR / source.name, dirs_exist_ok=True)

shutil.copy2(BASE_CONFIG, BACKUP_DIR / BASE_CONFIG.name)
shutil.copy2(DATA_CONFIG, BACKUP_DIR / DATA_CONFIG.name)

print(f"Đã sao lưu checkpoints, memory bank, log và config tại {BACKUP_DIR}")